# The Source Separation Algorithm
--------------------------------

##### Imports

In [4]:
%load_ext autoreload
%autoreload 2

import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd
from IPython.display import display
import ipywidgets as widgets
import soundfile as sf
import scipy.io.wavfile as wavfile

# Add the src directory to Python path to import our modules
import sys
sys.path.append('src')
sys.path.append('.')
from data_io import load_audio, save_audio
from dsp import compute_stft
from nmf_engine import fit_nmf
from reconstruction import generate_masks, apply_masks_and_reconstruct, refine_masks
from clustering import cluster_components, Source_Clustering, cluster_components_spectral

print('Modules loaded successfully! Run the next cell to show the dashboard.')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Modules loaded successfully! Run the next cell to show the dashboard.


###### blackbox

In [5]:
# 1. Interactive File Selection
data_dirs = ['data', '.', '..', 'C:/Users/user/OneDrive/Source Separation Project/Complete Audio files']
available_files = []
for d in data_dirs:
    if os.path.exists(d):
        available_files.extend(glob.glob(os.path.join(d, '*.wav')))
        available_files.extend(glob.glob(os.path.join(d, '*.flac')))

# Remove duplicates and format nicely
available_files = list(set([os.path.abspath(f) for f in available_files]))

file_dropdown = widgets.Dropdown(
    options=[(os.path.basename(f), f) for f in available_files],
    description='Audio File:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

start_time = widgets.Text(value='0:00', description='Start (M:S):', layout=widgets.Layout(width='200px'))
end_time = widgets.Text(value='end', description='End (M:S):', layout=widgets.Layout(width='200px'))
win_size_44 = widgets.IntText(value=1024, description='Win Size 44k:', layout=widgets.Layout(width='200px'))
win_size_96 = widgets.IntText(value=1024, description='Win Size 96k:', layout=widgets.Layout(width='200px'))
n_components = widgets.IntText(value=10, description='NMF Components:', layout=widgets.Layout(width='200px'))
n_sources = widgets.IntText(value=2, description='Final Sources:', layout=widgets.Layout(width='200px'))
clustering_method = widgets.Dropdown(
    options=['Agglomerative', 'K-Means', 'Affinity'],
    value='Affinity',
    description='Clustering:',
    layout=widgets.Layout(width='200px')
)
temporal_weight = widgets.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.1, description='Temp Wght:', layout=widgets.Layout(width='200px'))
mask_type = widgets.Dropdown(options=['soft', 'hard'], value='soft', description='Mask Type:', layout=widgets.Layout(width='200px'))
mask_power = widgets.IntSlider(value=2, min=1, max=4, description='Mask Power:', layout=widgets.Layout(width='200px'))
smooth_kernel = widgets.IntSlider(value=5, min=1, max=15, step=2, description='Smooth K:', layout=widgets.Layout(width='200px'))
floor_threshold = widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.01, description='Floor Cut:', layout=widgets.Layout(width='200px'))
transition_width = widgets.FloatSlider(value=0.05, min=0.001, max=0.2, step=0.01, description='Trans. W:', layout=widgets.Layout(width='200px'))
temporal_weight.layout.display = 'none'
def on_clustering_change(change):
    if change['new'] in ['Spectral', 'Affinity']:
        temporal_weight.layout.display = 'flex'
    else:
        temporal_weight.layout.display = 'none'
clustering_method.observe(on_clustering_change, names='value')
cost_function = widgets.Dropdown(
    options=['frobenius', 'kullback-leibler', 'itakura-saito'],
    value='kullback-leibler',
    description='Cost Func:',
    layout=widgets.Layout(width='200px')
)
save_checkbox = widgets.Checkbox(value=False, description='Save Output Files', indent=False)
run_button = widgets.Button(description="Load and Separate", button_style='success', icon='play')

output_area = widgets.Output()

def parse_time(time_str):
    """Converts MM:SS or seconds string to a float in seconds."""
    if not time_str.strip() or time_str.strip().lower() in ['none', 'all', 'full', 'end']:
        return None
    try:
        if ':' in time_str:
            parts = time_str.split(':')
            if len(parts) == 2:
                return float(parts[0]) * 60 + float(parts[1])
        return float(time_str)
    except ValueError:
        return "invalid"

# 2. Visualization and Playback Function
import re
import json
import glob
import shutil

def process_audio(b):
    with output_area:
        output_area.clear_output(wait=True)
        filepath = file_dropdown.value
        if not filepath:
            print("Please select a file.")
            return
            
        start_sec = parse_time(start_time.value)
        end_sec = parse_time(end_time.value)
        
        if start_sec == "invalid" or end_sec == "invalid":
            print("Invalid time format. Please use numbers or MM:SS (e.g. 1:05)")
            return
            
        # 1. Parse filename and find both copies
        basename = os.path.basename(filepath)
        dirpath = os.path.dirname(filepath)
        
        file_prefix = basename
        abs_start_sec = 0
        abs_end_sec = 0
        has_time_tag = False
        
        # Regex for '(XmYs_to_AmBs)'
        time_match = re.search(r'\((\d+)m(\d+)s_to_(\d+)m(\d+)s\)', basename)
        if time_match:
            has_time_tag = True
            m1, s1, m2, s2 = map(int, time_match.groups())
            abs_start_sec = m1 * 60 + s1
            abs_end_sec = m2 * 60 + s2
            
            # strip time and khz tags to get clean prefix
            file_prefix = re.sub(r'_(96kHz|44kHz)_.+', '', basename)
            file_prefix = re.sub(r'\.wav$', '', file_prefix)
        else:
            file_prefix = re.sub(r'(_)?(96kHz|44\.1kHz|44kHz)', '', basename)
            file_prefix = re.sub(r'\.wav$', '', file_prefix)

        # calculate specific absolute times for this run
        run_start_abs = abs_start_sec + (start_sec if start_sec else 0)
        dur = (end_sec - start_sec) if (start_sec is not None and end_sec is not None) else 0
        run_end_abs = run_start_abs + dur if dur > 0 else (abs_end_sec if has_time_tag else 0)
        
        start_tag = f"{int(run_start_abs//60)}m{int(run_start_abs%60)}s"
        end_tag = f"{int(run_end_abs//60)}m{int(run_end_abs%60)}s" if run_end_abs > 0 else "end"
        
        out_parent_name = f"{file_prefix}_{start_tag}_to_{end_tag}"
        
        # finding twin files
        if '96kHz' in basename:
            filepath_96 = filepath
            filepath_44 = os.path.join(dirpath, basename.replace('96kHz', '44kHz'))
            if not os.path.exists(filepath_44): # fallback
                filepath_44 = os.path.join(dirpath, basename.replace('96kHz', '44.1kHz'))
        elif '44kHz' in basename or '44.1kHz' in basename:
            filepath_44 = filepath
            filepath_96 = os.path.join(dirpath, basename.replace('44kHz', '96kHz').replace('44.1kHz', '96kHz'))
        else:
            print(f"Filename must contain _44kHz or _96kHz tag. Couldn't parse: {basename}")
            return
            
        if not os.path.exists(filepath_44) or not os.path.exists(filepath_96):
            print(f"Error: Could not find both 44kHz and 96kHz files natively.")
            print(f"Looked for: {filepath_44}")
            print(f"Looked for: {filepath_96}")
            return
            
        print(f"Found paired files for: {file_prefix}")
        print(f"Absolute time range: {start_tag} to {end_tag}")
        
        # Setup runs folder
        out_root = "output_sources"
        parent_dir = os.path.join(out_root, out_parent_name)
        
        run_base_name = f"{n_components.value} Comp {n_sources.value} Sour"
        if not os.path.exists(parent_dir):
            run_num = 1
        else:
            existing = [d for d in os.listdir(parent_dir) if d.startswith(f"{run_base_name} [Run ")]
            run_num = 1
            for e in existing:
                match = re.search(r'\[Run (\d+)\]', e)
                if match:
                    run_num = max(run_num, int(match.group(1)) + 1)
        
        run_dir = os.path.join(parent_dir, f"{run_base_name} [Run {run_num}]")
        
        n_fft_44 = int(win_size_44.value)
        hop_44 = n_fft_44 // 4
        n_fft_96 = int(win_size_96.value)
        hop_96 = n_fft_96 // 4
        
        import json
        if save_checkbox.value:
            os.makedirs(parent_dir, exist_ok=True)
            os.makedirs(run_dir, exist_ok=True)
            os.makedirs(os.path.join(run_dir, f"{file_prefix}_96"), exist_ok=True)
            os.makedirs(os.path.join(run_dir, f"{file_prefix}_44"), exist_ok=True)
            
            # Save settings JSON
            settings = {
                "n_components": n_components.value,
                "n_sources": n_sources.value,
                "beta_loss": cost_function.value,
                "mask_type": mask_type.value,
                "mask_power": mask_power.value,
                "smooth_kernel": smooth_kernel.value,
                "floor_threshold": floor_threshold.value,
                "transition_width": transition_width.value,
                "clustering_method": clustering_method.value,
                "temporal_weight": temporal_weight.value,
                "n_fft_44": n_fft_44,
                "hop_length_44": hop_44,
                "n_fft_96": n_fft_96,
                "hop_length_96": hop_96
            }
            with open(os.path.join(run_dir, "settings.json"), "w") as f:
                json.dump(settings, f, indent=4)
        
        # Dual-loop
        targets = [
            (44100, filepath_44, n_fft_44, hop_44, "44"),
            (96000, filepath_96, n_fft_96, hop_96, "96")
        ]
        
        for tgt_sr, tgt_fp, tgt_n_fft, tgt_hop, tgt_tag in targets:
            print(f"\n============================================================")
            print(f"### Processing {tgt_tag}kHz pipeline ###")
            print(f"============================================================")
            
            try:
                y, sr = load_audio(tgt_fp, start_sec=start_sec, end_sec=end_sec)
            except Exception as e:
                print(f"Error loading {tgt_tag}kHz audio: {e}")
                continue
                
            if save_checkbox.value:
                save_audio(os.path.join(parent_dir, f"original_cut_{tgt_tag}kHz.wav"), y, sr)
                
            print(f"Computing STFT (n_fft={tgt_n_fft}, hop={tgt_hop})...")
            window = 'hann'
            D = compute_stft(y, n_fft=tgt_n_fft, hop_length=tgt_hop, window=window)
            V_mag = np.abs(D)
            
            # Display Original Spectrogram
            plt.figure(figsize=(8, 4))
            librosa.display.specshow(librosa.amplitude_to_db(V_mag, ref=np.max),
                                     y_axis='log', x_axis='time', sr=sr, hop_length=tgt_hop)
            plt.title(f'Original Spectrogram - {file_prefix} ({tgt_tag}kHz)')
            plt.colorbar(label='dB')
            plt.tight_layout()
            if save_checkbox.value:
                plt.savefig(os.path.join(parent_dir, f"spect_original_{file_prefix}_{tgt_tag}kHz.png"))
            plt.show()
            
            print(f"Original Mixed Audio ({tgt_tag}kHz):")
            display(ipd.Audio(data=y, rate=sr))
            
            print(f"\nRunning NMF Separation ({n_components.value} components)...")
            W, H = fit_nmf(V_mag, n_components=n_components.value, max_iter=50, use_custom=False, beta_loss=cost_function.value)
            
            print(f"Clustering into {n_sources.value} final sources...")
            if clustering_method.value == 'Agglomerative':
                W_clustered, H_clustered = cluster_components(W, H, n_sources=n_sources.value, sr=sr, n_fft=tgt_n_fft)
            elif clustering_method.value in ['Spectral', 'Affinity']:
                W_clustered, H_clustered = cluster_components_spectral(W, H, n_sources=n_sources.value, sr=sr, temporal_weight=temporal_weight.value)
            else:
                bases, activations = Source_Clustering(W, H, inst_num=n_sources.value, sr=sr)
                W_clustered = np.zeros((W.shape[0], n_sources.value))
                H_clustered = np.zeros((n_sources.value, H.shape[1]))
                for j in range(n_sources.value):
                    W_clustered[:, j] = np.sum(bases[j], axis=1)
                    H_clustered[j, :] = np.sum(activations[j], axis=0)
            
            print("Generating Masks & Reconstructing...")
            masks = generate_masks(W_clustered, H_clustered, mask_type=mask_type.value, power=mask_power.value)
            masks = refine_masks(masks, smooth_kernel=smooth_kernel.value, floor_threshold=floor_threshold.value, transition_width=transition_width.value)
            estimated_sources_list = apply_masks_and_reconstruct(D, masks, hop_length=tgt_hop, length=len(y))
            
            for i, source in enumerate(estimated_sources_list):
                print(f"Source {i+1} ({tgt_tag}kHz):")
                
                # Spectrogram for Source
                S_db = librosa.amplitude_to_db(np.abs(compute_stft(source, n_fft=tgt_n_fft, hop_length=tgt_hop, window=window)), ref=np.max)
                plt.figure(figsize=(8, 4))
                librosa.display.specshow(S_db, y_axis='log', x_axis='time', sr=sr, hop_length=tgt_hop)
                plt.title(f'Source {i+1} - {file_prefix} ({tgt_tag}kHz)')
                plt.colorbar(label='dB')
                plt.tight_layout()
                if save_checkbox.value:
                    spect_path = os.path.join(run_dir, f"{file_prefix}_{tgt_tag}", f"spect_source{i+1}_{file_prefix}_{tgt_tag}kHz.png")
                    plt.savefig(spect_path)
                plt.show()
                
                display(ipd.Audio(data=source, rate=sr))
                
                if save_checkbox.value:
                    out_path = os.path.join(run_dir, f"{file_prefix}_{tgt_tag}", f"source_{i+1}_{tgt_tag}kHz.wav")
                    save_audio(out_path, source, sr)
                    print(f"Saved: {out_path}")
                
            print(f"--- Completed {tgt_tag}kHz ---")
        
        print("\n=== ALL SEPARATIONS FINISHED ===")

run_button.on_click(process_audio)


#### Dashboard

In [6]:
# Display Dashboard Layout
if True:
    edit_masking_checkbox = widgets.Checkbox(value=False, description='Edit Masking', indent=False)
    masking_box = widgets.VBox([
        widgets.HBox([mask_type, mask_power, smooth_kernel]),
        widgets.HBox([floor_threshold, transition_width])
    ])
    masking_box.layout.display = 'none'

    def on_masking_toggle(change):
        if change['new']:
            masking_box.layout.display = 'flex'
        else:
            masking_box.layout.display = 'none'
            
    edit_masking_checkbox.observe(on_masking_toggle, names='value')

    controls_box = widgets.VBox([
        file_dropdown,
        widgets.HBox([start_time, end_time, cost_function]),
        widgets.HBox([win_size_44, win_size_96]),
        widgets.HBox([n_components, n_sources, clustering_method, temporal_weight]),
        edit_masking_checkbox,
        masking_box,
        widgets.HBox([save_checkbox])
    ])
    ui = widgets.VBox([controls_box, run_button, output_area])
    display(ui)
